# day-21-agent-hands-on — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [7]:
# ---- Solution 2 ----
ORDERS = {"5512": {"status": "refund_approved", "date": "2024-03-02"}}
def get_order_status(order_id):
    return ORDERS.get(str(order_id), {"error": "order not found"})

def two_tool_planner(question, history):
    calls = [h[0] for h in history]
    if "get_order_status" not in calls:
        oid = re.search(r"\b(\d{3,6})\b", question)
        if oid:
            return {"type": "action", "tool": "get_order_status", "input": {"order_id": oid.group(1)}}
    if "search_kb" not in calls:
        return {"type": "action", "tool": "search_kb", "input": {"query": "how long do refunds take"}}
    order = next((h[2] for h in history if h[0] == "get_order_status"), {})
    kb = next((h[2] for h in history if h[0] == "search_kb"), [])
    return {"type": "final", "text": f"Order status: {order.get('status')}. "
            + (kb[0]["text"] if kb else "")}

def run_multi(question, planner, max_steps=5):
    TOOLS = {"search_kb": lambda **k: search_kb(**k), "get_order_status": lambda **k: get_order_status(**k)}
    history = []
    for _ in range(max_steps):
        act = planner(question, history)
        if act["type"] == "final": return act["text"]
        history.append((act["tool"], act["input"], TOOLS[act["tool"]](**act["input"])))
    return "stopped"

print(run_multi("did my refund for order 5512 go through, and how long should it take?", two_tool_planner))

Order status: refund_approved. Refunds are issued to the original payment method within 5 business days of approval. Digital goods are non-refundable once downloaded.


In [8]:
# ---- Solution 5 ----
def run_cost(hops, pin=1/1e6, pout=5/1e6):
    total = 0.0
    for h in range(hops):
        in_tok = 900 + 400 * h
        total += in_tok * pin + 150 * pout
    return total
c1, c3 = run_cost(1), run_cost(3)
print(f"S5: 1-hop ${c1:.5f}/query, 3-hop ${c3:.5f}/query ({c3/c1:.1f}x)")
print(f"    at 100k/month: 1-hop ${c1*100_000:,.0f}, 3-hop ${c3*100_000:,.0f}, delta ${(c3-c1)*100_000:,.0f}")

S5: 1-hop $0.00165/query, 3-hop $0.00615/query (3.7x)
    at 100k/month: 1-hop $165, 3-hop $615, delta $450


### Solutions 1, 3, 4, 6 (sketch)

**S1:** paste `run_agent_anthropic` from §2, call it in the §3 loop. Expect the model to make
0 tool calls for "hi", 1 for simple questions, 2 for the compare/multi questions — same shape
as the local planner but with better query phrasing and abstention wording, and real latency
+ token cost you can now measure.

**S3:** raising the abstention floor to 0.35 usually converts a borderline OOS "answer" into a
clean abstention with no loss on real KB questions (whose top scores are typically 0.45+).
Tune it on your eval like Day 18's `min_vscore`.

**S4:** *"I was charged after I cancelled — what should have happened?"* has no "and", so the
splitter yields one part and the agent does one `search_kb` (probably hitting `cancel`). A
real model would recognise two sub-questions ("what does cancellation do to billing?" +
"what's the refund path for an erroneous charge?") and search twice — planning, not pattern-
matching on conjunctions.

**S6:** add `if total_searches >= max_searches: return abstain/summarize` and keep the
repeat-count check. The looping planner trips the repeat guard at step 3; the 6-distinct-
search planner trips `max_searches=4` at step 5. Either way the run is bounded.

### Answer key
1. Whether to retrieve at all, how many times, and with what query — decided per request by
   the model instead of fixed by you.
2. `resp.stop_reason`: `"tool_use"` means execute the tool(s) and loop; anything else
   (`"end_turn"`, …) means the final answer is in the response's text blocks.
3. So the model can decide to abstain instead of answering from a weak match — the score is
   only useful if the model knows how to interpret it.
4. `max_steps` (bounds iterations / cost); repeated-query / no-progress detection (catches
   loops the model can't see); also a total search budget and per-tool timeout.
5. When queries vary in what they need — greetings, single-fact, multi-hop, out-of-scope — so
   a fixed one-retrieval pipeline is wasteful for some and insufficient for others.
6. The repeated-action guard (`counts[query] > max_same`) — it aborts once the same search
   query is issued more than the allowed number of times.
7. The keyword splitter treats it as one part and does a single search, likely missing the
   second lookup. A real model plans from the semantics of the question and issues two
   searches.